<a href="https://colab.research.google.com/github/adityas9450z-hash/HCL-Project/blob/main/Hospital_Appointment_Patient_Flow_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hospital Appointment and Patient Flow Analysis
### Patient Data Analysis of 40 Appointments

This project analyses the appointment data of 40 patients of a hospital. It is an **operations analysis** project, not a medical diagnosis project. We study appointments, cancellations, missed visits and waiting times to find where scheduling is not efficient and where patients may face delays.

**Questions**
1. How many appointments were scheduled, completed, cancelled and missed?
2. What is the missed appointment rate overall and by department?
3. Which weekdays and time windows have the highest demand?
4. Which departments have longer average or median waiting time?
5. How does waiting time change in busy and quiet periods?
6. Is the time between booking and appointment related to attendance?
7. Are there differences in appointment patterns across age groups?
8. What changes can reduce waiting time and unused slots?

Libraries used: only **NumPy** and **Pandas**.

## Step 1: Import libraries

In [ ]:
import numpy as np
import pandas as pd

## Step 2: Create the dataset
We do not have real patient data, so we create a sample dataset of **40 patients** using NumPy. Each row is one patient's appointment. The last line saves the data as a CSV file.

In [ ]:
np.random.seed(1)
n = 40
df = pd.DataFrame()
df["appointment_id"] = np.arange(1, n + 1)
df["age"] = np.random.randint(1, 90, n)
df["department"] = np.random.choice(["General", "Cardiology", "Orthopedics", "Pediatrics"], n)
df["weekday"] = np.random.choice(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"], n)
df["hour"] = np.random.randint(9, 17, n)
df["lead_days"] = np.random.randint(0, 30, n)
df["status"] = np.random.choice(["Completed", "Cancelled", "Missed"], n, p=[0.7, 0.1, 0.2])
df["wait_minutes"] = np.random.randint(10, 60, n)
df.loc[df["status"] != "Completed", "wait_minutes"] = np.nan   # no waiting time if patient did not come
df.to_csv("hospital_patient_flow_dataset.csv", index=False)

## Step 3: Look at the data

In [ ]:
print(df.head())
print("Rows and columns:", df.shape)
print(df.isnull().sum())

   appointment_id  age   department   weekday  hour  lead_days     status  \
0               1   38   Cardiology    Friday     9         13  Completed   
1               2   13   Pediatrics  Thursday    11         17  Completed   
2               3   73   Pediatrics   Tuesday    13         18  Completed   
3               4   10  Orthopedics  Thursday    13          1  Completed   
4               5   76   Cardiology    Monday     9         21     Missed   

   wait_minutes  
0          42.0  
1          29.0  
2          13.0  
3          34.0  
4           NaN  
Rows and columns: (40, 8)
appointment_id    0
age               0
department        0
weekday           0
hour              0
lead_days         0
status            0
wait_minutes      9
dtype: int64


`wait_minutes` is empty for cancelled and missed visits because those patients did not wait. This is normal.

## Q1. How many appointments were scheduled, completed, cancelled and missed?

In [ ]:
print("Total appointments:", len(df))
print(df["status"].value_counts())

Total appointments: 40
status
Completed    31
Missed        5
Cancelled     4
Name: count, dtype: int64


## Q2. Missed appointment rate (overall and by department)

In [ ]:
df["missed"] = df["status"] == "Missed"
print("Overall missed rate (%):", round(df["missed"].mean() * 100, 1))
print("Missed rate (%) by department:")
print((df.groupby("department")["missed"].mean() * 100).round(1))

Overall missed rate (%): 12.5
Missed rate (%) by department:
department
Cardiology     27.3
General        11.1
Orthopedics    12.5
Pediatrics      0.0
Name: missed, dtype: float64


## Q3. Weekdays and time windows with highest demand

In [ ]:
print("Appointments by weekday:")
print(df["weekday"].value_counts())
print("Appointments by hour:")
print(df["hour"].value_counts().sort_index())

Appointments by weekday:
weekday
Friday       10
Monday        9
Thursday      8
Tuesday       7
Wednesday     6
Name: count, dtype: int64
Appointments by hour:
hour
9     6
10    5
11    8
12    1
13    7
14    2
15    7
16    4
Name: count, dtype: int64


## Q4. Departments with longer waiting times

In [ ]:
completed = df[df["status"] == "Completed"]
print("Waiting time (minutes) by department:")
print(completed.groupby("department")["wait_minutes"].agg(["mean", "median"]).round(1))

Waiting time (minutes) by department:
             mean  median
department               
Cardiology   36.0    38.0
General      40.2    41.0
Orthopedics  40.3    42.0
Pediatrics   36.9    40.0


## Q5. Waiting time in busy and quiet periods

In [ ]:
print("Average wait by hour:")
print(completed.groupby("hour")["wait_minutes"].mean().round(1))
print("Average wait by weekday:")
print(completed.groupby("weekday")["wait_minutes"].mean().round(1))

Average wait by hour:
hour
9     31.7
10    36.3
11    39.6
12    59.0
13    36.5
14    54.5
15    37.7
16    33.0
Name: wait_minutes, dtype: float64
Average wait by weekday:
weekday
Friday       33.4
Monday       34.7
Thursday     36.8
Tuesday      46.9
Wednesday    40.8
Name: wait_minutes, dtype: float64


## Q6. Is booking gap related to attendance?

In [ ]:
df["long_booking"] = df["lead_days"] > 15
print("Missed rate (%) - True = booked more than 15 days ahead:")
print((df.groupby("long_booking")["missed"].mean() * 100).round(1))
print("Average booking gap (days) for each status:")
print(df.groupby("status")["lead_days"].mean().round(1))

Missed rate (%) - True = booked more than 15 days ahead:
long_booking
False     0.0
True     22.7
Name: missed, dtype: float64
Average booking gap (days) for each status:
status
Cancelled    10.5
Completed    15.4
Missed       21.4
Name: lead_days, dtype: float64


## Q7. Age groups

In [ ]:
# Child = below 18, Adult = 18 to 59, Senior = 60 and above
df["age_group"] = np.where(df["age"] < 18, "Child", np.where(df["age"] < 60, "Adult", "Senior"))
print("Patients in each age group:")
print(df["age_group"].value_counts())
print("Missed rate (%) by age group:")
print((df.groupby("age_group")["missed"].mean() * 100).round(1))

Patients in each age group:
age_group
Senior    16
Child     14
Adult     10
Name: count, dtype: int64
Missed rate (%) by age group:
age_group
Adult     10.0
Child     14.3
Senior    12.5
Name: missed, dtype: float64


## Q8. Key findings

In [ ]:
print("Department with highest missed rate:", df.groupby("department")["missed"].mean().idxmax())
print("Department with longest average wait:", completed.groupby("department")["wait_minutes"].mean().idxmax())
print("Busiest weekday:", df["weekday"].value_counts().idxmax())
print("Busiest hour:", df["hour"].value_counts().idxmax())
print("Average age of patients:", round(df["age"].mean(), 1))

Department with highest missed rate: Cardiology
Department with longest average wait: Orthopedics
Busiest weekday: Friday
Busiest hour: 11
Average age of patients: 42.2


### Suggestions to reduce waiting and unused slots
1. **Reminders:** send SMS or call reminders one or two days before the appointment, mainly for departments with a high missed rate.
2. **Small overbooking:** where many visits are missed, book a few extra patients so empty slots are used.
3. **Shorter booking gap:** if patients who book far ahead miss more, confirm those bookings again and keep some slots for same-week booking.
4. **Spread the demand:** move routine visits from the busiest weekday and hour to quieter ones.
5. **More staff at peak times:** add doctors or nurses in the busiest hours.
6. **Check long-wait departments:** review slot length and doctor schedule in the department with the longest waiting time.
7. **Waiting list:** keep a list of patients who can come at short notice when someone cancels.

**Note:** This is sample data of only 40 patients, so the numbers are only for practice. With real hospital data the same code can be used to get real results.

## Conclusion
In this project we used NumPy and Pandas to study the appointments of 40 patients. We counted completed, cancelled and missed appointments, found the missed rate, the busiest weekdays and hours, the waiting time of each department, and compared age groups. The main ways to improve are reminders for missed visits, spreading patients across days and hours, and reducing waiting in the busiest department.